# EDA — Bài A: Fraud / Error Detection (Xóm Bank)

Phân tích khám phá dữ liệu giao dịch từ `BankingDW` phục vụ mô hình phát hiện giao dịch lỗi.

**Target:** `is_error` = 1 khi `errors IS NOT NULL` (giao dịch lỗi). Mất cân bằng ~1.74%.

> Ghi chú hạn chế: DW chỉ lưu tới **ngày** (không có giờ giao dịch), nên feature 'hour of day' — thường mạnh cho fraud — không khả dụng ở tầng DW.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from db import read_fact_dataset

df = read_fact_dataset()
print(df.shape)
df.head()

## 1. Phân phối target (mất cân bằng)

In [ ]:
rate = df['is_error'].mean()
print(f'Tỷ lệ lỗi: {rate:.4f}  ({df["is_error"].sum():,} / {len(df):,})')
df['is_error'].value_counts().plot(kind='bar', title='0 = thành công, 1 = lỗi')
plt.show()

## 2. Phân phối Amount theo lỗi / thành công

Kỳ vọng: giao dịch lỗi (đặc biệt Insufficient Balance) có thể lệch về giá trị cao hơn.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for val, lbl in [(0, 'Thành công'), (1, 'Lỗi')]:
    sub = df.loc[df['is_error'] == val, 'amount'].clip(-100, 500)
    ax[0].hist(sub, bins=60, alpha=0.5, density=True, label=lbl)
ax[0].legend(); ax[0].set_title('Phân phối Amount (density)')
df.boxplot(column='amount', by='is_error', ax=ax[1])
ax[1].set_ylim(-50, 300); ax[1].set_title('Amount theo is_error')
plt.suptitle(''); plt.tight_layout(); plt.show()

print(df.groupby('is_error')['amount'].describe())

## 3. Tỷ lệ lỗi theo các chiều phân loại

Entry mode, card type, MCC, state — chiều nào phân hóa rủi ro mạnh nhất?

In [ ]:
def error_rate_by(col, top=10):
    g = df.groupby(col)['is_error'].agg(['mean', 'count'])
    g = g[g['count'] >= 100].sort_values('mean', ascending=False)
    return g.head(top)

for col in ['entry_mode', 'card_type', 'card_brand']:
    print(f'\n=== Tỷ lệ lỗi theo {col} ===')
    print(error_rate_by(col))

In [ ]:
# Entry mode: kênh Online thường lỗi cao hơn
er = df.groupby('entry_mode')['is_error'].mean().sort_values(ascending=False)
er.plot(kind='bar', title='Tỷ lệ lỗi theo Entry Mode'); plt.ylabel('error rate'); plt.show()

print('\nTop 10 MCC theo tỷ lệ lỗi:')
print(error_rate_by('mcc_description', 10))

## 4. Tương quan giữa các feature số

In [ ]:
num = ['amount', 'current_age', 'yearly_income', 'credit_score',
       'debt_to_income', 'num_credit_cards', 'credit_limit', 'is_error']
corr = df[num].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num, rotation=45, ha='right')
ax.set_yticks(range(len(num))); ax.set_yticklabels(num)
plt.colorbar(im); plt.title('Ma trận tương quan'); plt.tight_layout(); plt.show()

print('Tương quan với is_error:')
print(corr['is_error'].sort_values(ascending=False))

## 5. Kết luận EDA

- Target rất mất cân bằng (1.74%) → dùng PR-AUC, SMOTE/scale_pos_weight, KHÔNG dùng accuracy.
- `amount` là tín hiệu mạnh nhất (khớp SHAP): lỗi Insufficient Balance gắn với số tiền.
- Kênh **Online** có tỷ lệ lỗi cao hơn Chip/Swipe → `entry_mode` hữu ích.
- Tương quan tuyến tính yếu → mô hình phi tuyến (XGBoost) phù hợp hơn LogReg.
- **Hạn chế:** thiếu giờ giao dịch ở DW; nếu cần nâng recall, bổ sung `hour`, `velocity` (số GD/thẻ/ngày) từ `BankingDB.transactions`.

Chi tiết huấn luyện & đánh giá: `src/train_fraud.py`, `src/evaluate.py`. Kết quả: `models/`.